# SDXL Fair Inference — Colab Demo

Randomly selects a `female` or `male` concept per seed and generates:
- an **original** image (no concept injection)
- a **concept** image (concept injected at the SDXL UNet mid-block)

Make sure you're on a GPU runtime: **Runtime → Change runtime type → T4/A100/L4**.

## 1. Install dependencies

In [ ]:
!pip install -q diffusers transformers accelerate safetensors

## 2. Clone the repo (brings the checkpoints with it)

Pretrained `concept_female.pt`, `concept_male.pt`, and `concept_dict.json` live in
the repo's `checkpoints/` folder, so a clone gives us everything we need.

In [ ]:
!git clone -q https://github.com/Moslem-Sh21/learning-where-it-matters.git /content/lwim
!ls -lh /content/lwim/checkpoints/

## 3. Imports

In [ ]:
import os
import json
import random
from tqdm.auto import tqdm
import copy

import torch
from diffusers import (
    DDPMScheduler,
    AutoencoderKL,
    StableDiffusionXLPipeline,
    UNet2DConditionModel,
)
from transformers import CLIPTextModel, CLIPTextModelWithProjection, CLIPTokenizer

from PIL import Image
import matplotlib.pyplot as plt

assert torch.cuda.is_available(), "No GPU detected. Switch to a GPU runtime."
print("GPU:", torch.cuda.get_device_name(0))

## 4. Configuration

Edit anything here. Paths assume you uploaded the three files in step 2.

In [ ]:
# Model
PRETRAINED          = "stabilityai/stable-diffusion-xl-base-1.0"

# Uploaded checkpoints
FEMALE_CONCEPT_PATH = "/content/lwim/checkpoints/concept_female.pt"
MALE_CONCEPT_PATH   = "/content/lwim/checkpoints/concept_male.pt"
CONCEPT_DICT_PATH   = "/content/lwim/checkpoints/concept_dict.json"

# Generation
PROMPT              = "a cinematic shot of a senior executive leading a tense boardroom meeting"
NUM_IMAGES          = 4
GUIDANCE_SCALE      = 7.5
CONCEPT_SCALE       = 2.0
RESOLUTION          = 1024
NUM_INFERENCE_STEPS = 30
FP16                = True

# Bookkeeping
OUTPUT_DIR          = "/content/outputs"
SEEDS_FILE          = "/content/generated_seeds.json"
MAX_CONCEPT_LENGTH  = 100

DEVICE       = "cuda"
WEIGHT_DTYPE = torch.float16 if FP16 else torch.float32

os.makedirs(os.path.join(OUTPUT_DIR, "original"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "concept"),  exist_ok=True)

## 5. `BottleneckConceptModule` + global concept state

In [ ]:
# Global one-hot store. The mid-block hook reads from this dict at every forward pass.
CURRENT_INPUT_CONDITIONS = {"tensor": None}


class BottleneckConceptModule(torch.nn.Module):
    """
    Learnable concept vectors added at the SDXL UNet mid-block.
    num_concepts : e.g. 100
    mid_channels : 1280 for SDXL
    """

    def __init__(self, num_concepts: int, mid_channels: int):
        super().__init__()
        self.num_concepts = num_concepts
        self.mid_channels = mid_channels
        self.concept_emb = torch.nn.Parameter(torch.zeros(num_concepts, mid_channels))
        torch.nn.init.normal_(self.concept_emb, mean=0.0, std=0.02)

    def forward(self, mid_tensor: torch.Tensor, concept_onehot: torch.Tensor) -> torch.Tensor:
        if concept_onehot is None:
            return mid_tensor
        B, C, H, W = mid_tensor.shape
        # Match the dtype/device of the mid-block tensor so this works under
        # both fp32 and fp16 (concept_emb is cast along with the UNet).
        concept_onehot = concept_onehot.to(device=mid_tensor.device, dtype=mid_tensor.dtype)
        if concept_onehot.shape[0] != B:
            concept_onehot = concept_onehot.expand(B, -1)
        delta = (concept_onehot @ self.concept_emb).view(B, C, 1, 1)
        return mid_tensor + delta

## 6. Helpers: seeds + UNet/concept loader

In [ ]:
def get_or_create_seeds(seeds_file: str, num_seeds: int) -> list:
    if os.path.exists(seeds_file):
        with open(seeds_file) as f:
            seeds = json.load(f)["seeds"]
        print(f"Loaded {len(seeds)} seeds from '{seeds_file}'")
    else:
        seeds = random.sample(range(1, 1_000_000), num_seeds)
        with open(seeds_file, "w") as f:
            json.dump({"seeds": seeds}, f)
        print(f"Generated and saved {len(seeds)} seeds -> '{seeds_file}'")
    return seeds[:num_seeds]


def load_unet_with_concept(
        base_unet: UNet2DConditionModel,
        concept_ckpt_path: str,
        concept_dict_path: str,
        concept_name: str,
        max_concept_length: int,
        concept_scale: float,
        device: str,
):
    """
    Deep-copies the HF-pretrained base_unet (so we can attach independent concept
    modules + hooks per gender), loads the concept module, and registers the
    mid-block hook gated by CURRENT_INPUT_CONDITIONS.

    Returns:
        unet         - ready UNet on `device`
        concept_cond - one-hot tensor [1, max_concept_length] on `device`
    """
    unet = copy.deepcopy(base_unet)

    mid_channels = unet.config.block_out_channels[-1]  # 1280 for SDXL
    concept_module = BottleneckConceptModule(
        num_concepts=max_concept_length, mid_channels=mid_channels
    )
    print(f"  Concept: {concept_ckpt_path}")
    concept_module.load_state_dict(torch.load(concept_ckpt_path, map_location="cpu"))
    unet.concept_module = concept_module

    def mid_block_concept_hook(module, inputs, output):
        onehot = CURRENT_INPUT_CONDITIONS["tensor"]
        if onehot is None:
            return output
        out = unet.concept_module(output, onehot)
        return output + concept_scale * (out - output)

    unet.mid_block.register_forward_hook(mid_block_concept_hook)

    with open(concept_dict_path) as f:
        concept_dict = json.load(f)
    if concept_name not in concept_dict:
        raise ValueError(
            f"Concept '{concept_name}' not found in {concept_dict_path}. "
            f"Available keys: {list(concept_dict.keys())}"
        )
    concept_index = concept_dict[concept_name]
    print(f"  Concept key '{concept_name}' -> index {concept_index}")

    concept_cond = torch.zeros(1, max_concept_length, device=device)
    concept_cond[:, concept_index] = 1.0

    unet.to(device)
    return unet, concept_cond

## 7. Load shared SDXL components

In [ ]:
print("Loading shared SDXL components from HF...")

tokenizer_one    = CLIPTokenizer.from_pretrained(PRETRAINED, subfolder="tokenizer")
tokenizer_two    = CLIPTokenizer.from_pretrained(PRETRAINED, subfolder="tokenizer_2")
text_encoder_one = CLIPTextModel.from_pretrained(PRETRAINED, subfolder="text_encoder")
text_encoder_two = CLIPTextModelWithProjection.from_pretrained(PRETRAINED, subfolder="text_encoder_2")
vae              = AutoencoderKL.from_pretrained(PRETRAINED, subfolder="vae")
scheduler        = DDPMScheduler.from_pretrained(PRETRAINED, subfolder="scheduler")

print("Loading pretrained UNet from HF (no local checkpoint)...")
base_unet = UNet2DConditionModel.from_pretrained(PRETRAINED, subfolder="unet")
print("Done.")

## 8. Load female and male UNets

In [ ]:
print("Loading female UNet + concept module...")
female_unet, female_cond = load_unet_with_concept(
    base_unet,
    concept_ckpt_path=FEMALE_CONCEPT_PATH,
    concept_dict_path=CONCEPT_DICT_PATH,
    concept_name="female",
    max_concept_length=MAX_CONCEPT_LENGTH,
    concept_scale=CONCEPT_SCALE,
    device=DEVICE,
)

print("\nLoading male UNet + concept module...")
male_unet, male_cond = load_unet_with_concept(
    base_unet,
    concept_ckpt_path=MALE_CONCEPT_PATH,
    concept_dict_path=CONCEPT_DICT_PATH,
    concept_name="male",
    max_concept_length=MAX_CONCEPT_LENGTH,
    concept_scale=CONCEPT_SCALE,
    device=DEVICE,
)

## 9. Build the pipeline

In [ ]:
pipe = StableDiffusionXLPipeline(
    vae=vae,
    text_encoder=text_encoder_one,
    text_encoder_2=text_encoder_two,
    tokenizer=tokenizer_one,
    tokenizer_2=tokenizer_two,
    unet=female_unet,  # placeholder; swapped per image
    scheduler=scheduler,
).to(DEVICE)

pipe.vae.to(dtype=WEIGHT_DTYPE)
pipe.text_encoder.to(dtype=WEIGHT_DTYPE)
pipe.text_encoder_2.to(dtype=WEIGHT_DTYPE)
female_unet.to(dtype=WEIGHT_DTYPE)   # concept_module rides along with the UNet cast
male_unet.to(dtype=WEIGHT_DTYPE)
pipe.safety_checker = None

SIZE_KWARGS = dict(
    original_size=(RESOLUTION, RESOLUTION),
    target_size=(RESOLUTION, RESOLUTION),
    crops_coords_top_left=(0, 0),
)
print("Pipeline ready.")

## 10. Quick example — 2 seeds, female & male

A small reproducible showcase before the bulk run: two fixed seeds, each
generating an `original` (no concept) and a `concept` version (one with
`female`, one with `male`). Same seed → same noise, so any visual difference
comes from the concept injection at the mid-block.

In [ ]:
# Two fixed seeds for a reproducible demo
EXAMPLE_SEEDS   = [42, 1234]
EXAMPLE_GENDERS = ["female", "male"]   # one concept per seed

example_results = []

for seed, gender in zip(EXAMPLE_SEEDS, EXAMPLE_GENDERS):
    if gender == "female":
        pipe.unet, cond = female_unet, female_cond
    else:
        pipe.unet, cond = male_unet,   male_cond

    # 1) ORIGINAL (no concept) — same seed
    CURRENT_INPUT_CONDITIONS["tensor"] = None
    gen  = torch.Generator(device=DEVICE).manual_seed(seed)
    orig = pipe(
        prompt=PROMPT,
        generator=gen,
        guidance_scale=GUIDANCE_SCALE,
        num_inference_steps=NUM_INFERENCE_STEPS,
        **SIZE_KWARGS,
    ).images[0]

    # 2) CONCEPT — same seed
    CURRENT_INPUT_CONDITIONS["tensor"] = cond
    gen  = torch.Generator(device=DEVICE).manual_seed(seed)
    conc = pipe(
        prompt=PROMPT,
        generator=gen,
        guidance_scale=GUIDANCE_SCALE,
        num_inference_steps=NUM_INFERENCE_STEPS,
        **SIZE_KWARGS,
    ).images[0]

    example_results.append({
        "seed": seed, "gender": gender, "original": orig, "concept": conc,
    })

# 2x2 grid: rows = seeds, cols = (original, concept)
fig, axes = plt.subplots(2, 2, figsize=(10, 10))
for row, r in enumerate(example_results):
    axes[row, 0].imshow(r["original"]); axes[row, 0].axis("off")
    axes[row, 0].set_title(f"original  (seed {r['seed']})")
    axes[row, 1].imshow(r["concept"]);  axes[row, 1].axis("off")
    axes[row, 1].set_title(f"concept = {r['gender']}  (seed {r['seed']})")

plt.tight_layout()
plt.show()

## 11. Bulk generation loop

In [ ]:
seeds = get_or_create_seeds(SEEDS_FILE, NUM_IMAGES)

dir_original = os.path.join(OUTPUT_DIR, "original")
dir_concept  = os.path.join(OUTPUT_DIR, "concept")

print(f"Prompt         : {PROMPT!r}")
print(f"Num images     : {NUM_IMAGES}")
print(f"Concept scale  : {CONCEPT_SCALE}")
print(f"Guidance scale : {GUIDANCE_SCALE}\n")

metadata = []

for seed in tqdm(seeds, desc="Generating"):

    chosen_gender = random.choice(["female", "male"])

    if chosen_gender == "female":
        pipe.unet    = female_unet
        concept_cond = female_cond
    else:
        pipe.unet    = male_unet
        concept_cond = male_cond

    # 1) ORIGINAL (no concept)
    CURRENT_INPUT_CONDITIONS["tensor"] = None
    gen = torch.Generator(device=DEVICE).manual_seed(seed)
    out_original = pipe(
        prompt=PROMPT,
        generator=gen,
        guidance_scale=GUIDANCE_SCALE,
        num_inference_steps=NUM_INFERENCE_STEPS,
        **SIZE_KWARGS,
    )
    orig_path = os.path.join(dir_original, f"seed_{seed}.jpg")
    out_original.images[0].save(orig_path)

    # 2) CONCEPT (chosen gender)
    CURRENT_INPUT_CONDITIONS["tensor"] = concept_cond
    gen = torch.Generator(device=DEVICE).manual_seed(seed)
    out_concept = pipe(
        prompt=PROMPT,
        generator=gen,
        guidance_scale=GUIDANCE_SCALE,
        num_inference_steps=NUM_INFERENCE_STEPS,
        **SIZE_KWARGS,
    )
    conc_path = os.path.join(dir_concept, f"seed_{seed}_{chosen_gender}.jpg")
    out_concept.images[0].save(conc_path)

    metadata.append({
        "seed": seed,
        "concept": chosen_gender,
        "original_file": orig_path,
        "concept_file":  conc_path,
    })

meta_path = os.path.join(OUTPUT_DIR, "metadata.json")
with open(meta_path, "w") as f:
    json.dump(metadata, f, indent=2)

female_count = sum(1 for m in metadata if m["concept"] == "female")
male_count   = len(metadata) - female_count
print(f"\nDone! {len(metadata)} seeds. female: {female_count}, male: {male_count}")
print(f"Metadata: {meta_path}")

## 12. Preview all bulk results — original vs concept, side by side

In [ ]:
n = len(metadata)
fig, axes = plt.subplots(n, 2, figsize=(10, 5 * n))
if n == 1:
    axes = axes.reshape(1, 2)

for row, m in enumerate(metadata):
    img_o = Image.open(m["original_file"])
    img_c = Image.open(m["concept_file"])
    axes[row, 0].imshow(img_o); axes[row, 0].axis("off")
    axes[row, 0].set_title(f"original (seed {m['seed']})")
    axes[row, 1].imshow(img_c); axes[row, 1].axis("off")
    axes[row, 1].set_title(f"concept = {m['concept']} (seed {m['seed']})")

plt.tight_layout()
plt.show()